In [1]:
import base64
import datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Preprocessing Functions

In [2]:
def decode_base64(item):
    return base64.b64decode(item.encode()).decode('utf-8')

In [3]:
def decode_human_ai_hash(df):
    df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])
    df = df.applymap(lambda x: x.replace('\xa0', '').strip() if isinstance(x, str) else x)
    nan_summary = df.isnull().sum()
    print("Number of NaN values in each column:\n", nan_summary)
    for column in df.columns:
        if df[column].isnull().sum() > 0:
            majority_value = df[column].mode()[0]
            df[column].fillna(majority_value, inplace = True)
    return df

In [4]:
participants = ['p1', 'p2', 'p3', 'p4']

human_assessed_requirements = [pd.read_excel(f'./{i}_human_evaluation/task_d/task_d.xlsx') for i in participants]
human_assessed_requirements = [decode_human_ai_hash(df) for df in human_assessed_requirements]

Number of NaN values in each column:
 component                                                                                                0
zephyr_7b_beta_missing_requirments                                                                       0
This requirement is consistent with other project requirements.                                          0
This requirement accurately reflects the needs that were previously unstated, missing, or overlooked.    0
Including this requirement would lead to a more complete set of project specifications.                  0
dtype: int64
Number of NaN values in each column:
 component                                                                                                0
zephyr_7b_beta_missing_requirments                                                                       0
This requirement is consistent with other project requirements.                                          0
This requirement accurately reflects the needs that wer

/var/folders/sl/2p60p6g94jzf57rhnp2dng1h0000gn/T/ipykernel_95348/3822606932.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\xa0', '').strip() if isinstance(x, str) else x)
/var/folders/sl/2p60p6g94jzf57rhnp2dng1h0000gn/T/ipykernel_95348/3822606932.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\xa0', '').strip() if isinstance(x, str) else x)
/var/folders/sl/2p60p6g94jzf57rhnp2dng1h0000gn/T/ipykernel_95348/3822606932.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\xa0', '').strip() if isinstance(x, str) else x)
/var/folders/sl/2p60p6g94jzf57rhnp2dng1h0000gn/T/ipykernel_95348/3822606932.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\xa0', '').strip() if isinstance(x, s

In [5]:
component_label2id = {'self-evaluation': 0, 'adaptation functionality': 1, 'chat-bot': 2}
likert_label2id = {'Strongly Disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly Agree': 5}

likert_id2label = {v: k for k, v in zip(likert_label2id.keys(), likert_label2id.values())}
component_id2label = {v: k for k, v in zip(component_label2id.keys(), component_label2id.values())}

In [6]:
human_assessed_requirements_concated = pd.concat(human_assessed_requirements, axis = 0)

compoenent = human_assessed_requirements_concated.columns[0]
consistency, refelect_needs_missing, lead_to_more_complet = human_assessed_requirements_concated.columns[2:]

In [7]:
human_assessed_requirements_concated = pd.concat(human_assessed_requirements, axis = 0, ignore_index = True)

compoenent = human_assessed_requirements_concated.columns[0]
requirement = human_assessed_requirements_concated.columns[1]
consistency, refelect_needs_missing, lead_to_more_complet = human_assessed_requirements_concated.columns[2:]

# Convert Likert ratings to numeric values
for column in [consistency, refelect_needs_missing, lead_to_more_complet]:
    human_assessed_requirements_concated[column] = human_assessed_requirements_concated[column].map(likert_label2id)

ratings_per_requirement = human_assessed_requirements_concated.groupby([compoenent, requirement]).size()

# Aggregate the four ratings using the median
human_assessed_requirements_concated = (human_assessed_requirements_concated
                                        .groupby([compoenent, requirement], as_index = False)
                                        .agg({consistency: 'median', refelect_needs_missing: 'median', lead_to_more_complet: 'median'})
                                       )

print('Number of aggregated requirements:', len(human_assessed_requirements_concated))

print('\nRequirements per component:')
print(human_assessed_requirements_concated[compoenent].value_counts())

Number of aggregated requirements: 32

Requirements per component:
component
self-evaluation             12
adaptation functionality    11
chat-bot                     9
Name: count, dtype: int64


# Inferential Statistics

## Why Wilcoxon Signed-Rank Test Selected?

The one-sample Wilcoxon signed-rank test is recommended over the one-sample t-test and the sign test for your Likert data, given its strengths in handling non-normal, ordinal data. Usman et al. note that while the t-test is powerful under normal distributions, it is less suitable when normality is in doubt, which can often be the case with Likert data [1]. The Wilcoxon test is preferred over the sign test as it accounts for both the direction and magnitude of differences, offering greater sensitivity and power, as highlighted by Divine et al. [2]. Arnold’s findings further indicate that, compared to the t-test, the Wilcoxon test maintains higher power for non-normal distributions, making it a robust choice for ordinal data [3]. Lastly, Randles emphasizes the Wilcoxon test’s efficiency compared to the t-test when data deviate from normality, underscoring its suitability for ordinal Likert scales [4].

## References
[1] Usman, M. "Power Efficiency of Sign Test and Wilcoxon Signed Rank Test Relative to T-Test", Mathematical Theory and Modeling, 2015.<br>
[2] Divine, G. et al. "A Review of Analysis and Sample Size Calculation Considerations for Wilcoxon Tests", Anesthesia & Analgesia, 2013.<br>
[3] Arnold, H. J. "Small Sample Power of the One Sample Wilcoxon Test for Non-Normal Shift Alternatives", The Annals of Mathematical Statistics, 1965.<br>
[4] Randles, R. H. "Wilcoxon Signed Rank Test", Encyclopedia of Statistical Sciences, 2006.

In [8]:
# Rank Biserial effect size
import stikpetP as ps

# def rank_biserial(data, hypMed):
#     data = pd.Series(data)
#     return ps.r_rank_biserial_os(data, mu = hypMed)['rb'][0]
from scipy.stats import rankdata

def rank_biserial(data, hypMed=3):
    differences = np.asarray(data, dtype=float) - hypMed
    ranks = rankdata(np.abs(differences))

    zero_ranks = ranks[differences == 0].sum()
    positive_ranks = ranks[differences > 0].sum() + zero_ranks / 2
    negative_ranks = ranks[differences < 0].sum() + zero_ranks / 2

    return (
        (positive_ranks - negative_ranks)
        / (positive_ranks + negative_ranks)
    )

In [9]:
# Wilcoxon Test and Bootstrap CI
from scipy.stats import wilcoxon
from scipy.stats import bootstrap


def wilcoxon_rank_test(x, hypMed = 3, ci = 0.95):
    x_median = x.median()
    x_mean = x.mean()
    sample_size = len(x)
    print('Sample_ ', sample_size)
    differences = np.array(x) - hypMed
    test_statistic, p_value = wilcoxon(differences, alternative = "greater", zero_method = 'zsplit') # less -> left-tailed, use zero_method pratt, zsplit, wilcox
    
    print(f"p_value: {p_value:.5f}")
    print("Wilcoxon statistic:", test_statistic)

    print(f'\nMedian of the responses {x_median}')
    print(f'Mean of the responses {x_mean}')
    
    std_dev = np.std(x, ddof=1)
    print(f'Standard Deviation {std_dev}')
    
    rbc = rank_biserial(x, hypMed)
    print(f"\neffect_size_rbc: {rbc:.5f}")

    resampled_medians = bootstrap((x,), lambda data: rank_biserial(data, hypMed), confidence_level = ci, n_resamples = 1000, method = 'percentile', random_state = 42)
    ci_lower, ci_upper = resampled_medians.confidence_interval
    print(f"\n{int(ci * 100)}% Rank Biserial Correlation CI: ({ci_lower:.5f}, {ci_upper:.5f})")
    return p_value

## **Variable Name:** Consistent with Requirements Set$_{(CRS)}$
**Variable Description:** This requirement is consistent with other project requirements.

### **Hypotheses:**
- $H_{0,7}$: The median rating ($M$) for the Consistent with Requirements Set$_{(CRS)}$ is  $\leq 3$.
- **$H_{a,7}$:** The median rating for Consistent with Requirements Set$_{(CRS)}$ is $> 3$.

In [10]:
consistent = human_assessed_requirements_concated.loc[:, 'This requirement is consistent with other project requirements.']
p_value_1 = wilcoxon_rank_test(consistent)

Sample_  32
p_value: 0.00000
Wilcoxon statistic: 517.0

Median of the responses 4.5
Mean of the responses 4.1875
Standard Deviation 0.6444552588389371

effect_size_rbc: 0.95833

95% Rank Biserial Correlation CI: (0.85985, 1.00000)


## **Variable Name:** Identify Missing Requirements$_{(IMR)}$
**Variable Description:** This requirement accurately reflects the needs that were previously unstated, missing, or overlooked

### **Hypotheses:**
- **$H_{0,8}$:** The median rating ($M$) for the Identify Missing Requirements$_{(IMR)}$ is $\leq 3$.
- **$H_{a,8}$:** The median rating for Identify Missing Requirements$_{(IMR)}$ is $> 3$.

In [11]:
missing = human_assessed_requirements_concated.loc[:, 'This requirement accurately reflects the needs that were previously unstated, missing, or overlooked.']
p_value_2 = wilcoxon_rank_test(missing)

Sample_  32
p_value: 0.00003
Wilcoxon statistic: 474.5

Median of the responses 4.0
Mean of the responses 3.890625
Standard Deviation 0.8105590961719551

effect_size_rbc: 0.79735

95% Rank Biserial Correlation CI: (0.53594, 0.98864)


## **Variable Name:** Enhancing the Overall Completeness$_{(EOC)}$
**Variable Description:** Including this requirement would lead to a more complete set of project specifications.

### **Hypotheses:**
- **$H_{0,9}$:** The median rating ($M$) for the Enhancing the Overall Completeness$_{(EOC)}$ is $\leq 3$.
- **$H_{a,9}$:** The median rating for Enhancing the Overall Completeness$_{(EOC)}$ is $> 3$

In [12]:
complete = human_assessed_requirements_concated.loc[:, 'Including this requirement would lead to a more complete set of project specifications.']
p_value_3 = wilcoxon_rank_test(complete)

Sample_  32
p_value: 0.00007
Wilcoxon statistic: 463.5

Median of the responses 4.0
Mean of the responses 3.734375
Standard Deviation 0.8326442043080237

effect_size_rbc: 0.75568

95% Rank Biserial Correlation CI: (0.48097, 0.93750)


# **Holm-Bonferroni Correction**

## **Report:** 
All the addjusted p-values using Holm-Bonferroni correction method is confirming the un-adjusted p-values.

In [13]:
from statsmodels.stats.multitest import multipletests

# Example list of p-values
pvals = [p_value_1, p_value_2, p_value_3]

# Perform Holm-Bonferroni correction
reject, pvals_corrected, _, _ = multipletests(pvals, alpha = 0.05, method = 'holm')

print("Original p-values:", [f"{i:.5f}" for i in pvals])
print("Adjusted p-values:", [f"{i:.5f}" for i in pvals_corrected])
print("Reject null hypothesis:", reject)

Original p-values: ['0.00000', '0.00003', '0.00007']
Adjusted p-values: ['0.00000', '0.00006', '0.00007']
Reject null hypothesis: [ True  True  True]
